# Unpack shared data
Original student archive includes 80 training and 10 student-test trips, no evaluator trips. Verify checksums and extract once. Originals are retained unchanged.

In [ ]:
from pathlib import Path
import os
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
assert (ROOT / "data").is_dir(), "Open this notebook from the repository root or notebooks directory"
os.chdir(ROOT)


In [ ]:
import json, hashlib, zipfile
checks = json.loads(Path("data/SHA256SUMS.json").read_text())
for archive, expected in checks.items():
    parts = sorted(Path(archive).parent.glob(Path(archive).name + ".part*"))
    if parts:
        with Path(archive).open("wb") as destination:
            for part in parts:
                destination.write(part.read_bytes())
    assert Path(archive).exists(), f"Archive/parts missing: {archive}"
    actual = hashlib.sha256(Path(archive).read_bytes()).hexdigest()
    assert actual == expected, f"Checksum mismatch: {archive}"
    with zipfile.ZipFile(archive) as z:
        for member in z.namelist():
            target = (ROOT / member).resolve()
            assert target.is_relative_to(ROOT.resolve()), "Unsafe archive path"
        z.extractall(ROOT)
print("Ready: 80 training trips, 10 student-test trips, and shared preprocessed arrays.")
